<a href="https://colab.research.google.com/github/namii07/Namisha-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_03_ETL_Pandas_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests --quiet
# The --quiet flag supresses the installation o/p
import pandas as pd

import numpy as np

import requests
# requests: The standard python library for making http api calls

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

print('All Libraries imported successfully!')
print(f'pandas : {pd.__version__}')
print(f'requests : {requests.__version__}')

All Libraries imported successfully!
pandas : 2.2.2
requests : 2.32.4


PART 1: ETL on Messy Sales Data

Activity 1 - Clean messy_sales_data.csv

Known data quality issues in this file :
1. Missing values in customer_name, quantity, category
2. Duplicate rows (orders 1001/1005 are identical)
3. Mixed Date formats : YYYY-MM-DD and DD-MM-YYYY
4. Inconsistent text case in customer_name (UPPER, lower, Title)
5. Wrong category value (keyboard labelled as Electronics)

EXTARCT: LOAD THE RAW DATA

In [2]:
raw_df = pd.read_csv('messy_sales_data.csv')
# we store the raw data in "raw_df"
# ETL best practice:: NEVER modify raw data
print("Number of Rows:",  raw_df.shape[0])
print("Number of Columns:", raw_df.shape[1])
print("Columns in the dataset")
print(raw_df.columns.tolist())
raw_df.head()

Number of Rows: 30
Number of Columns: 9
Columns in the dataset
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [3]:
# Diagnose All data quality problems before  fixing

print('=' *55)
print(' DATA QUALITY DIAGNOSIS REPORT')
print('=' *55)

print('\n[1] MISSING VALUES per column')
print(raw_df.isnull().sum())

print(f'\n[2] DUPLICATE ROWS:{ raw_df.duplicated().sum()}')

print(f'\n[3] DATA TYPES:')
print(raw_df.dtypes)

print('\n[4] UNIQUE CATEGORIES:', raw_df['category'].unique())
print('[4] Sample customer names:', raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_date values:' , raw_df['order_date'].unique()[:6])

 DATA QUALITY DIAGNOSIS REPORT

[1] MISSING VALUES per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS:0

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [4]:
df = raw_df.copy()

print(f'Working copy created : {df.shape}')
print('raw_df is untouched - we can always reset by running df = raw_df.copy()')

Working copy created : (30, 9)
raw_df is untouched - we can always reset by running df = raw_df.copy()


In [5]:
print('Before fixing nulls:' , df.isnull().sum().sum(), 'total missing values')
df['customer_name'].fillna('Unknown Customer', inplace=True)
median_qty  = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f' Filled missing  quantity with median: {median_qty}')
df['category'].fillna('Uncategorised', inplace=True)
print('After fixing nulls:' , df.isnull().sum().sum(), 'total missing values')

Before fixing nulls: 7 total missing values
 Filled missing  quantity with median: 2.0
After fixing nulls: 1 total missing values


In [6]:
print(f'Before deduplication:{len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])
df.drop_duplicates(inplace=True)
print(f'\nAfter deduplication:{len(df)} rows')
print(f'Rows removed: {len(raw_df) - len(df)}')

Before deduplication:30 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After deduplication:30 rows
Rows removed: 0


In [7]:
print("Sample dates before parsing : ")
print(df['order_date'].head(8).tolist())

df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
    )

nat_count=df['order_date'].isnull().sum()
print(f"\nUnparseable dates (NaT) : {nat_count}")

df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
#.strftime('%B) returns the full month name

print("\nSample dates after parsing : ")
print(df[['order_date','year', 'month', 'month_name']].head(5))

Sample dates before parsing : 
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Unparseable dates (NaT) : 2

Sample dates after parsing : 
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [8]:

print("Before standardiztion : ", df['customer_name'].unique()[:6])

df['customer_name']=(
    df['customer_name']
    .str.strip()
    .str.title()
)

print("After standardiztion : ", df['customer_name'].unique()[:6])

print(f"\nBefore : keyboard rows with Electronics category : ")
wrong_mask=(df['product']=='keyboard')&(df['category']=='Electronics')

df.loc[wrong_mask, 'category']='Accessories'

print("After fix : Unique categories : ",df['category'].unique())

Before standardiztion :  ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After standardiztion :  ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']

Before : keyboard rows with Electronics category : 
After fix : Unique categories :  ['Electronics' 'Accessories' 'Uncategorised']


In [9]:
df['quantity']=pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price']=pd.to_numeric(df['unit_price'], errors='coerce')
#.to_numeric converts the column to a numeric type
#errors='coerce' turns unconvertible values to NaN instead of crashing

df['revenue']=df['quantity']*df['unit_price']

print("Revenue column created : ")
print(df[['customer_name', 'product', 'quantity', 'unit_price', 'revenue']].head(5))
print(f"\nTotal revenue : {df['revenue'].sum():,.0f} Rs")

Revenue column created : 
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    Amit Verma  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0

Total revenue : 818,000 Rs


In [10]:

print("=" * 55)
print("    POST - CLEANING VALIDATION REPORT")
print("=" * 55)

print(f"Original rows : {len(raw_df)}")
print(f"Cleaned rows : {len(df)}")
print(f"Rows removed : {len(raw_df)-len(df)} (duplicates)")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Duplicates : {df.duplicated().sum()}")
print(f"Date nulls : {df['order_date'].isnull().sum()}")
print(f"Revenue NaN : {df['revenue'].isnull().sum()}")
print(f"Categories : {sorted(df['category'].unique())}")

print("=" * 55)

all_clean=(
    df.isnull().sum().sum()==0 and
    df.duplicated().sum()==0
)
print(f"DATA IS CLEAN : {all_clean}")

    POST - CLEANING VALIDATION REPORT
Original rows : 30
Cleaned rows : 30
Rows removed : 0 (duplicates)
Missing values : 9
Duplicates : 0
Date nulls : 2
Revenue NaN : 0
Categories : ['Accessories', 'Electronics', 'Uncategorised']
DATA IS CLEAN : False


In [11]:

product_rev=(
    df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)

print("Revenue by product:")
print(product_rev.to_string(index=False))

category_summary=df.groupby('category').agg(
    total_revenue =('revenue','sum'),
    avg_order_value =('revenue','mean'),
    num_orders=('order_id','count'),
    unique_products=('product','nunique')
).round(2).reset_index()
#agg() applies multiple functions to multiple columns in one call
#nunique -> count of unique values (not built into groupby directly)

print("\n Category summary:")
print(category_summary.to_string(index=False))

Revenue by product:
   product  revenue
    Laptop 540000.0
   Monitor 154000.0
Headphones  28000.0
     Mouse  20800.0
  Keyboard  20400.0
    Webcam  20000.0
   USB Hub  19800.0

 Category summary:
     category  total_revenue  avg_order_value  num_orders  unique_products
  Accessories        76200.0          5861.54          13                4
  Electronics       697800.0         43612.50          16                4
Uncategorised        44000.0         44000.00           1                1


In [12]:
df.to_csv('clean_sales_data.csv', index=False)
print("Cleaned data saved to clean_sales_data.csv")
print(f"Final dataset : {df.shape[0]} rows x {df.shape[1]} columns")
print("\nETL Pipeline for sales data : COMPLETE")
print(" EXTRACT -> messy_sales_data.csv loaded")
print(" TRANSFORM -> nulls fixed, dupes removed, dates parsed, names standardized")
print(" LOAD -> clean_sales_data.csv saved")

Cleaned data saved to clean_sales_data.csv
Final dataset : 30 rows x 13 columns

ETL Pipeline for sales data : COMPLETE
 EXTRACT -> messy_sales_data.csv loaded
 TRANSFORM -> nulls fixed, dupes removed, dates parsed, names standardized
 LOAD -> clean_sales_data.csv saved


In [13]:
API_KEY = '8714143d16fe59d1bda609543cdcd6fa '
BASE_URL = 'https://api.openweathermap.org/data/2.5/weather'

CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkatta', 'Pune', 'Jaipur']

print(f"API configured for {len(CITIES)} cities")
print(f"Cities : {CITIES}")
print("\n IMPORTANT : Replace YOUR_API_KEY_HERE with your actual key before running")

API configured for 8 cities
Cities : ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkatta', 'Pune', 'Jaipur']

 IMPORTANT : Replace YOUR_API_KEY_HERE with your actual key before running


PRACTICE QUESTIONS:

1. What are the three stages of ETL? Describe each stage using an example from today's sales dataset

2. A dataframe has 500 rows. After calling df.dropna() it has 412 rows. What does this tell you?

3. Write code to remove duplicates from df where 'same row' means same customer_name AND same product

4. What is the difference between fillna(0) AND fillna(df['col'].median())? When would you prefer each?

5. Write python code to call the weather API for delhi and print the temperature in celcius

6. What is response.status_code==200 mean? What should you do when the code is 401?

In [14]:
# Answer 1

"""
ETL stands for:

1. Extract
2. Transform
3. Load

1. Extract:
   Data is collected from a source such as CSV files, databases, or APIs.

   Example:
   Reading today's sales dataset using pandas.

"""

import pandas as pd

sales_data = pd.read_csv("messy_sales_data.csv")

print(sales_data.head())

"""
2. Transform:
   Data is cleaned and modified into a useful format.

   Example:
   - Removing missing values
   - Removing duplicates
   - Converting column datatypes
"""

sales_data = sales_data.dropna()
sales_data = sales_data.drop_duplicates()

"""
3. Load:
   The cleaned data is stored into another system such as a database,
   Excel file, or data warehouse.

   Example:
"""

sales_data.to_csv("cleaned_sales.csv", index=False)

print("ETL process completed successfully")

   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   
3      1004  Sunita Patel   Monitor  Electronics       NaN       22000   
4      1005  Ramesh Kumar    Laptop  Electronics       2.0       45000   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  
3  2024-01-10    Chennai   Ravi Kumar  
4  2024-01-05     Mumbai  Anil Sharma  
ETL process completed successfully


In [15]:

"""
The dataframe originally had 500 rows.

After using df.dropna(), only 412 rows remained.

This means:
500 - 412 = 88 rows contained missing (NULL/NaN) values.

dropna() removes all rows that contain at least one missing value.

Therefore, 88 rows were deleted because they had missing data.
"""

'\nThe dataframe originally had 500 rows.\n\nAfter using df.dropna(), only 412 rows remained.\n\nThis means:\n500 - 412 = 88 rows contained missing (NULL/NaN) values.\n\ndropna() removes all rows that contain at least one missing value.\n\nTherefore, 88 rows were deleted because they had missing data.\n'

In [16]:

import pandas as pd

df = df.drop_duplicates(subset=['customer_name', 'product'])

print(df)

    order_id     customer_name     product       category  quantity  \
0       1001      Ramesh Kumar      Laptop    Electronics       2.0   
1       1002        Priya Nair         NaN    Electronics       1.0   
2       1003        Amit Verma    Keyboard    Accessories       3.0   
3       1004      Sunita Patel     Monitor    Electronics       2.0   
5       1006       Kiran Mehta       Mouse    Accessories      10.0   
6       1007      Deepak Singh  Headphones    Electronics       2.0   
7       1008  Unknown Customer      Webcam    Accessories       1.0   
8       1009        Ananya Das      Laptop    Electronics       1.0   
9       1010       Vikram Iyer    Keyboard    Accessories       5.0   
10      1011       Pooja Gupta     Monitor    Electronics       2.0   
11      1012        Suresh Rao     USB Hub    Accessories       8.0   
12      1013       Meera Joshi      Laptop    Electronics       2.0   
13      1014        Arjun Nair  Headphones    Electronics       3.0   
14    

In [17]:
"""
fillna(0):
-----------
Replaces missing values with 0.

Example:
df['sales'] = df['sales'].fillna(0)

Use when:
- Missing value actually means zero
- Example: no sales, no profit, no discount


fillna(df['col'].median()):
---------------------------
Replaces missing values using the median of the column.

Example:
df['sales'] = df['sales'].fillna(df['sales'].median())

Use when:
- Data is numerical
- You do not want outliers to affect the replacement value
- Median is better for skewed data

Difference:
------------
fillna(0) inserts a fixed value (0)

fillna(median()) inserts a statistical value based on existing data
"""

"\nfillna(0):\n-----------\nReplaces missing values with 0.\n\nExample:\ndf['sales'] = df['sales'].fillna(0)\n\nUse when:\n- Missing value actually means zero\n- Example: no sales, no profit, no discount\n\n\nfillna(df['col'].median()):\n---------------------------\nReplaces missing values using the median of the column.\n\nExample:\ndf['sales'] = df['sales'].fillna(df['sales'].median())\n\nUse when:\n- Data is numerical\n- You do not want outliers to affect the replacement value\n- Median is better for skewed data\n\nDifference:\n------------\nfillna(0) inserts a fixed value (0)\n\nfillna(median()) inserts a statistical value based on existing data\n"

In [18]:

import requests

api_key = "8714143d16fe59d1bda609543cdcd6fa"

url = f"https://api.openweathermap.org/data/2.5/weather?q=Delhi&appid={api_key}&units=metric"

response = requests.get(url)

data = response.json()

print("Temperature in Delhi:", data['main']['temp'], "°C")

Temperature in Delhi: 28.05 °C


In [19]:
"""
response.status_code == 200 means:

The API request was successful.

The server understood the request and returned the required data successfully.


Example:
"""

import requests

response = requests.get("https://api.github.com")

print(response.status_code)

"""
If the status code is 401:

401 means "Unauthorized"

This happens when:
- API key is incorrect
- Token is missing
- User authentication failed

What should you do?
--------------------
1. Check whether the API key is correct
2. Verify login credentials
3. Make sure authentication token is valid
4. Ensure API permissions are enabled

Example:
"""

if response.status_code == 200:
    print("Request successful")

elif response.status_code == 401:
    print("Unauthorized access - Check API key or authentication")

200
Request successful


In [20]:
API_KEY = "6850fba354816244b7e81bbb9ef89bb2"
CITY = "Chennai"

# API URL
url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

# Send GET request
response = requests.get(url)

# Convert response to JSON
data = response.json()

# Display raw JSON data
print(data)

# Extract required fields
weather_data = {
    "City": [data["name"]],
    "Temperature (C)": [data["main"]["temp"]],
    "Feels Like (C)": [data["main"]["feels_like"]],
    "Humidity (%)": [data["main"]["humidity"]],
    "Pressure (hPa)": [data["main"]["pressure"]],
    "Weather": [data["weather"][0]["description"]],
    "Wind Speed (m/s)": [data["wind"]["speed"]]
}

# Create DataFrame
df = pd.DataFrame(weather_data)

# Display DataFrame
df

# Remove duplicates (if any)
df.drop_duplicates(inplace=True)

# Check for missing values
print(df.isnull().sum())

# Save to CSV
df.to_csv("weather_data.csv", index=False)

print("CSV file saved successfully!")

{'coord': {'lon': 80.2785, 'lat': 13.0878}, 'weather': [{'id': 701, 'main': 'Mist', 'description': 'mist', 'icon': '50d'}], 'base': 'stations', 'main': {'temp': 30.54, 'feels_like': 35.85, 'temp_min': 28.99, 'temp_max': 30.54, 'pressure': 1006, 'humidity': 68, 'sea_level': 1006, 'grnd_level': 1007}, 'visibility': 4000, 'wind': {'speed': 7.2, 'deg': 280}, 'clouds': {'all': 75}, 'dt': 1780025109, 'sys': {'type': 2, 'id': 2104103, 'country': 'IN', 'sunrise': 1780013498, 'sunset': 1780059672}, 'timezone': 19800, 'id': 1264527, 'name': 'Chennai', 'cod': 200}
City                0
Temperature (C)     0
Feels Like (C)      0
Humidity (%)        0
Pressure (hPa)      0
Weather             0
Wind Speed (m/s)    0
dtype: int64
CSV file saved successfully!
